In [5]:
# 📦 1) Gerekli Kütüphanelerin Yüklenmesi
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, logging
from IPython.display import HTML, display

logging.set_verbosity_error()

print("✅ Kütüphaneler başarıyla yüklendi!")

✅ Kütüphaneler başarıyla yüklendi!


In [6]:
# 🤖 2) Model ve Tokenizer Yükleme
device = "cuda"
model_id = "Qwen/Qwen3-0.6B"

print(f"📥 Model yükleniyor: {model_id}")
print(f"🖥️ Cihaz: {device}")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cuda"
).eval()

# Model katman sayısını al
num_layers = len(model.model.layers)
print(f"\n✅ Model başarıyla yüklendi!")
print(f"📊 Toplam katman sayısı: {num_layers}")

📥 Model yükleniyor: Qwen/Qwen3-0.6B
🖥️ Cihaz: cuda

✅ Model başarıyla yüklendi!
📊 Toplam katman sayısı: 28


In [7]:
# 🎯 3) Analiz Edilecek Katmanları Belirleme
# Alt katmanlar literal bilgi, üst katmanlar semantik bilgi taşır

LAYERS_TO_COMPARE = [
    (0, "Lowest (Embedding)"),      # Embedding'e yakın - ham token bilgisi
    (4, "Low (Lexical)"),           # Token düzeyinde - kelime yapısı
    (10, "Middle"),                 # Sözdizimsel - gramer yapıları
    (18, "Mid-High"),               # Geçiş katmanı
    (num_layers - 2, "Highest (Semantic)")  # Son katmanlar - anlam ve kavram
]

print("📋 Analiz edilecek katmanlar:")
print("-" * 40)
for idx, name in LAYERS_TO_COMPARE:
    print(f"  Layer {idx:2d}: {name}")

📋 Analiz edilecek katmanlar:
----------------------------------------
  Layer  0: Lowest (Embedding)
  Layer  4: Low (Lexical)
  Layer 10: Middle
  Layer 18: Mid-High
  Layer 26: Highest (Semantic)


In [8]:
# ✏️ 4) Prompt ve Hedef Kelime Tanımlama
prompt = "Freedom is the most fundamental right of every human being."
target_word = "freedom"

print(f"📝 Prompt: {prompt}")
print(f"🎯 Hedef kelime: '{target_word}'")

# Tokenize et
inputs = tokenizer(prompt, return_tensors="pt").to(device)
prompt_lower = prompt.lower()

# Token'ları decode et ve göster
token_ids = inputs["input_ids"][0].tolist()
decoded_tokens = [tokenizer.decode([tid], skip_special_tokens=True) for tid in token_ids]

print(f"\n🔤 Token sayısı: {len(decoded_tokens)}")
print(f"🔤 Tokens: {decoded_tokens}")

📝 Prompt: Freedom is the most fundamental right of every human being.
🎯 Hedef kelime: 'freedom'

🔤 Token sayısı: 11
🔤 Tokens: ['Freedom', ' is', ' the', ' most', ' fundamental', ' right', ' of', ' every', ' human', ' being', '.']


In [9]:
# 📍 5) Hedef Kelime Pozisyonunu Bulma
word_start = prompt_lower.find(target_word)
if word_start == -1:
    raise ValueError(f"Word '{target_word}' not found in prompt")
word_end = word_start + len(target_word)

print(f"📍 '{target_word}' kelimesinin pozisyonu:")
print(f"   Başlangıç: {word_start}")
print(f"   Bitiş: {word_end}")
print(f"\n📝 Görsel: '{prompt[:word_start]}[{prompt[word_start:word_end]}]{prompt[word_end:]}'")

📍 'freedom' kelimesinin pozisyonu:
   Başlangıç: 0
   Bitiş: 7

📝 Görsel: '[Freedom] is the most fundamental right of every human being.'


In [10]:
# 🎨 6) Renk Haritası Fonksiyonu
def heat_color(v):
    """
    Benzerlik değerine göre renk döndürür.
    
    Args:
        v: 0-1 arası benzerlik değeri
    
    Returns:
        Hex renk kodu
    """
    if v < 0.15:
        return "#e0e0e0"      # Açık gri - Çok düşük
    elif v < 0.3:
        return "#b0b0b0"      # Koyu gri - Düşük
    elif v < 0.5:
        return "#ffd166"      # Sarı - Orta
    elif v < 0.7:
        return "#f77f00"      # Turuncu - Yüksek
    else:
        return "#d62828"      # Kırmızı - Çok yüksek

# Renk skalasını göster
print("🎨 Renk Skalası:")
print("-" * 50)
scale_html = """
<div style='font-family: monospace; padding: 10px;'>
<span style='background:#e0e0e0; padding:5px 15px; margin:2px;'>Çok Düşük (&lt;0.15)</span>
<span style='background:#b0b0b0; padding:5px 15px; margin:2px;'>Düşük (0.15-0.3)</span>
<span style='background:#ffd166; padding:5px 15px; margin:2px;'>Orta (0.3-0.5)</span>
<span style='background:#f77f00; padding:5px 15px; margin:2px; color:#fff;'>Yüksek (0.5-0.7)</span>
<span style='background:#d62828; padding:5px 15px; margin:2px; color:#fff;'>Çok Yüksek (&gt;0.7)</span>
</div>
"""
display(HTML(scale_html))

🎨 Renk Skalası:
--------------------------------------------------


In [11]:
# 🔬 7) Katman Aktivasyonlarını Çıkarma ve Benzerlik Hesaplama
# Bu adım her katman için:
# 1. Forward pass ile aktivasyonları yakalar
# 2. Hedef kelime vektörünü bulur
# 3. Cosine similarity hesaplar

all_results = []

print("🔄 Katman analizi başlıyor...")
print("=" * 60)

for layer_idx, layer_name in LAYERS_TO_COMPARE:
    print(f"\n📊 Layer {layer_idx} ({layer_name}) işleniyor...")
    
    # Hook ile aktivasyon yakala
    activations = {}
    def hook_fn(module, inp, out):
        activations["resid"] = out.detach()
    
    hook = model.model.layers[layer_idx].register_forward_hook(hook_fn)
    
    # Forward pass
    with torch.no_grad():
        model(**inputs)
    
    hook.remove()
    acts = activations["resid"][0].float()
    print(f"   ✓ Aktivasyon boyutu: {acts.shape}")
    
    # Güvenli normalizasyon
    acts_norms = acts.norm(dim=1, keepdim=True)
    acts_norms = torch.clamp(acts_norms, min=1e-8)
    acts_norm = acts / acts_norms
    
    # Hedef kelime tokenlarını bul
    cursor = 0
    target_idxs = []
    for i, tok in enumerate(decoded_tokens):
        t = tok.strip()
        if not t:
            continue
        pos = prompt_lower.find(t.lower(), cursor)
        if pos == -1:
            for j in range(len(t), 0, -1):
                pos = prompt_lower.find(t[:j].lower(), cursor)
                if pos != -1:
                    break
        if pos == -1:
            continue
        end = pos + len(t)
        if not (end <= word_start or pos >= word_end):
            target_idxs.append(i)
        cursor = max(cursor + 1, end)
    
    if not target_idxs:
        target_idxs = list(range(len(decoded_tokens)))
    
    print(f"   ✓ Hedef token indeksleri: {target_idxs}")
    
    # Hedef vektör (ortalama)
    target_vec = acts[target_idxs].mean(dim=0)
    target_norm_val = target_vec.norm()
    if target_norm_val > 1e-8:
        target_norm = target_vec / target_norm_val
    else:
        target_norm = target_vec
    
    # Cosine similarity
    sims = torch.matmul(acts_norm, target_norm).cpu().numpy()
    sims = np.clip(sims, 0, None)
    max_sim = sims.max()
    if max_sim > 1e-8:
        sims = sims / max_sim
    
    raw_sims = sims.copy()
    
    # Token → karakter yayılımı
    char_vals = np.zeros(len(prompt))
    char_counts = np.zeros(len(prompt))
    
    cursor = 0
    for sim, tok in zip(sims, decoded_tokens):
        t = tok.strip()
        if not t:
            continue
        pos = prompt_lower.find(t.lower(), cursor)
        if pos == -1:
            for j in range(len(t), 0, -1):
                pos = prompt_lower.find(t[:j].lower(), cursor)
                if pos != -1:
                    break
        if pos == -1:
            continue
        end = pos + len(t)
        char_vals[pos:end] += sim
        char_counts[pos:end] += 1
        cursor = max(cursor + 1, end)
    
    char_vals = np.divide(
        char_vals, char_counts,
        out=np.zeros_like(char_vals),
        where=char_counts > 0
    )
    
    all_results.append((layer_idx, layer_name, char_vals, raw_sims))
    print(f"   ✓ Tamamlandı!")

print("\n" + "=" * 60)
print("✅ Tüm katmanlar başarıyla analiz edildi!")

🔄 Katman analizi başlıyor...

📊 Layer 0 (Lowest (Embedding)) işleniyor...
   ✓ Aktivasyon boyutu: torch.Size([11, 1024])
   ✓ Hedef token indeksleri: [0]
   ✓ Tamamlandı!

📊 Layer 4 (Low (Lexical)) işleniyor...
   ✓ Aktivasyon boyutu: torch.Size([11, 1024])
   ✓ Hedef token indeksleri: [0]
   ✓ Tamamlandı!

📊 Layer 10 (Middle) işleniyor...
   ✓ Aktivasyon boyutu: torch.Size([11, 1024])
   ✓ Hedef token indeksleri: [0]
   ✓ Tamamlandı!

📊 Layer 18 (Mid-High) işleniyor...
   ✓ Aktivasyon boyutu: torch.Size([11, 1024])
   ✓ Hedef token indeksleri: [0]
   ✓ Tamamlandı!

📊 Layer 26 (Highest (Semantic)) işleniyor...
   ✓ Aktivasyon boyutu: torch.Size([11, 1024])
   ✓ Hedef token indeksleri: [0]
   ✓ Tamamlandı!

✅ Tüm katmanlar başarıyla analiz edildi!


In [12]:
# 📈 8) Sonuçları Görselleştirme
html = "<div style='font-family: monospace; font-size: 14px;'>"
html += f"<h3>🎯 Hedef kelime: '{target_word}' (Soyut Kavram)</h3>"
html += f"<p><b>Prompt:</b> {prompt}</p>"
html += "<p><i>Alt katmanlar → literal benzerlik | Üst katmanlar → kavramsal/semantik benzerlik</i></p>"
html += "<hr>"

for layer_idx, layer_name, char_vals, raw_sims in all_results:
    # En yüksek benzerlikli tokenları bul
    top_3_idx = np.argsort(raw_sims)[-3:][::-1]
    top_tokens = [(decoded_tokens[i], raw_sims[i]) for i in top_3_idx if i < len(decoded_tokens)]
    top_str = ", ".join([f"'{t}':{v:.2f}" for t, v in top_tokens])
    
    html += f"<p><b>Layer {layer_idx}</b> ({layer_name}) - Top: {top_str}</p>"
    html += "<div style='line-height: 2; margin-bottom: 15px;'>"
    
    for ch, v in zip(prompt, char_vals):
        html += (
            f'<span style="background:{heat_color(v)};'
            f'color:#000; padding:3px 4px; margin:1px;'
            f'border-radius:4px; font-weight:500;">{ch}</span>'
        )
    
    html += "</div>"

html += "<hr>"

# Renk skalası
html += """
<div style='margin-top: 10px; font-size: 12px;'>
<b>Renk Skalası:</b> 
<span style='background:#e0e0e0; padding:2px 6px;'>Çok Düşük</span>
<span style='background:#b0b0b0; padding:2px 6px;'>Düşük</span>
<span style='background:#ffd166; padding:2px 6px;'>Orta</span>
<span style='background:#f77f00; padding:2px 6px; color:#fff;'>Yüksek</span>
<span style='background:#d62828; padding:2px 6px; color:#fff;'>Çok Yüksek</span>
</div>
"""

html += "</div>"

display(HTML(html))

## 📖 Beklenen Örüntü ve Yorumlama

### Alt Katmanlar (0-6)
- Sadece **"freedom"** kelimesi vurgulanmalı (literal eşleşme)
- Bu katmanlar token düzeyinde, yüzeysel bilgi taşır

### Üst Katmanlar (15+)
- **"right"**, **"human"**, **"fundamental"** kelimeleri de yüksek benzerlik göstermeli
- Bu kelimeler semantik olarak "özgürlük" kavramıyla ilişkili
- Üst katmanlar anlam ve kavramsal ilişkileri öğrenir

### Sonuç
Bu analiz, transformer modellerinin nasıl "anlam" oluşturduğunu gösterir:
- **Erken katmanlar:** Kelime yapısı, morfoloji
- **Orta katmanlar:** Sözdizimi, gramer
- **Son katmanlar:** Semantik, kavramsal ilişkiler